# **Predictive Modeling**

The goal of this file is to create a predictive model from Los Angeles Angels (LAA) data to estimate the expected price change in Kalshi sports trading markets.

## Data Description

### Features

- **d_price_w $\alpha$ _b $\beta$**: The double binned mean price at the time of a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **laa_homerun_dummy**: Binary indicator equal to 1 if LAA hit a home run at that timestamp, 0 otherwise.

- **runners_on_base**: The number of runners on base when the home run was hit.

- **inning**: The inning in which the home run was hit.

- **laa_score**: LAA's score when the home run was hit.

- **opp_score**: The opponent's score when the home run was hit.

- **score_delta**: LAA's score less the opponent's score when the home run was hit.

- **opponent_XYZ**: Opponent dummy variable; 1 if that team is playing LAA, 0 otherwise.

    - _XYZ_ is the abbreviated team name of the team LAA is playing.

- **d_rolling_std_ $\gamma$ trades_w $\alpha$ _b $\beta$**: The rolling standard deviation of  price change over a window defined by trade count. $$\sigma_{\gamma} = \sqrt{\frac{1}{\gamma - 1} \sum_{i=1}^{\gamma} \left( p_{i} - \overline{p} \right)^2}$$

    - $\gamma$ is the trade count which determines the window size: $\gamma \in \{2, 3, 4, \ldots, 49, 50\}$
    
    - Corresponds with the double binned mean price change and price
        - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
        - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **d_rolling_std_ $\lambda$ mins_w $\alpha$ _b $\beta$**: The rolling standard deviation of price change over a window defined by time (minutes). $$\sigma_{\lambda} = \sqrt{\frac{1}{n_\lambda - 1} \sum_{i=1}^{n_\lambda} \left( p_{i} - \overline{p} \right)^2}$$

    - $\lambda$ is the number of minutes which determines the window size: $\lambda \in \{2, 3, 4, \ldots, 49, 50\}$
    - $n_\lambda$ is the number of $p_{i}$ observations falling within the trailing $\lambda$-minute window.

    - Corresponds with the double binned mean price change and price
        - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
        - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

### Targets

- **d_px_chg_w $\alpha$ _b $\beta$**: The double binned mean price change measuring market reaction to a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

In [824]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from scipy.stats import norm
from datetime import timedelta
import plotly.graph_objects as go
from utils.data import DataProcessing

In [825]:
LAA_RED = '#BA0021'
LAA_NAVY = '#003263'
LAA_SILVER = '#C4CED4'

## NGBoost Model for Price Change Prediction

In [826]:
# Load in all NGBoost results across different double binned targets

ngb_dir_path = "../models/ngb/"

ngb_results = {}
for w in range(1, 7):
    for b in range(1, 7):
        file_name = f"model_d_px_chg_w{w}_b{b}"
        path = Path(ngb_dir_path) / f"{file_name}.pkl"
        try:
            with open(path, 'rb') as f:
                ngb_results[file_name] = pickle.load(f)
        except FileNotFoundError:
            print(f"Missing: {path}")

In [827]:
# Get summary statistics for plotting

ngb_model_stats = []
for k, v in ngb_results.items():
    ngb_model_stats.append({
        'y_col'       : k,
        'window'      : k.split('_')[-2][1],
        'bin'         : k.split('_')[-1][1],
        'test_r2'     : ngb_results[k]['best_test_r2'],
        'overfitting' : ngb_results[k]['best_overfitting']
    })

ngb_model_stats_df = pd.DataFrame(ngb_model_stats)

ngb_model_stats_df.sort_values('test_r2', ascending=False).head()

,y_col,window,bin,test_r2,overfitting
11,model_d_px_chg_w2_b6,2,6,0.848768,0.112535
16,model_d_px_chg_w3_b5,3,5,0.841689,0.115476
17,model_d_px_chg_w3_b6,3,6,0.820756,0.138600
21,model_d_px_chg_w4_b4,4,4,0.819451,0.132053
22,model_d_px_chg_w4_b5,4,5,0.815657,0.144701


In [828]:
# Plotting NGBoost models across all window and bin combinations

# Drop extreme outliers (one dropped)
ngb_model_stats_df = ngb_model_stats_df[ngb_model_stats_df['test_r2'] > 0.32]

laa_color_scale = [
    [0.0, LAA_NAVY],
    [0.5, LAA_SILVER],
    [1.0, LAA_RED]
]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=ngb_model_stats_df['window'],
    y=ngb_model_stats_df['bin'],
    z=ngb_model_stats_df['test_r2'],
    mode='markers',
    marker=dict(
        size=8,
        color=ngb_model_stats_df['overfitting'],
        colorscale=laa_color_scale,
        opacity=0.9,
        colorbar=dict(
            title=dict(
                text='Test R² - Train R²',
                font=dict(color=LAA_SILVER)
            ),
            tickfont=dict(color=LAA_SILVER)
        ),
        showscale=True
    ),
    text=[
        f"Window: {w}<br>Bin: {b}<br>Test R²: {r:.4f}<br>Overfitting: {m:.4f}" 
        for w, b, r, m in zip(
            ngb_model_stats_df['window'], 
            ngb_model_stats_df['bin'], 
            ngb_model_stats_df['test_r2'], 
            ngb_model_stats_df['overfitting']
            )
    ],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    title=dict(
        text='<b>Optimally Parameterized NGBoost Models (LAA)</b>',
        font={'size': 25, 'color': LAA_SILVER},
        x=0.5,
        xanchor='center',
        y=0.95,        
        yanchor='top' 
    ),
    scene=dict(
        xaxis_title='Window Size (α)',
        yaxis_title='Bin Size (β)',
        zaxis_title='Test R²',
        xaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        yaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        zaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        bgcolor='#111111',
        aspectmode='cube'
    ),
    width=1000,
    height=700,
    margin=dict(l=0, r=0, b=0, t=60),
    showlegend=False
)

fig.show()

### Parameter Analysis for a Strong Performing NGBoost Model

In [829]:
# Get strong performing NGB model, uses target with window = 4 and bin = 4

strong_ngb = ngb_results["model_d_px_chg_w4_b4"]

In [830]:
# Strong NGB statistics

print('Train MSE:', strong_ngb['best_train_mse'])
print('Test MSE:', strong_ngb['best_test_mse'])
print('Train R²:', strong_ngb['best_train_r2'])
print('Test R²:', strong_ngb['best_test_r2'])

Train MSE: 0.0014180914363891323
Test MSE: 0.004208290079032408
Train R²: 0.9515041201098973
Test R²: 0.8194508948436094


In [831]:
# Grid Search CV results for target with window = 4 and bin = 4

gscv_cols = [
    'rank_test_score',
    'param_learning_rate', 
    'param_minibatch_frac', 
    'param_n_estimators', 
    'mean_test_score', 
    'std_test_score',
    'mean_train_score', 
    'std_train_score'
]

strong_ngb['cv_results'][gscv_cols].sort_values('rank_test_score')

,rank_test_score,param_learning_rate,param_minibatch_frac,param_n_estimators,mean_test_score,std_test_score,mean_train_score,std_train_score
50,1,0.05,0.8,150,1.811458,2.733317,-2.341293,0.080906
53,2,0.05,1.0,150,1.681753,2.800605,-2.380091,0.077227
47,3,0.05,0.5,150,0.640115,1.703712,-2.117534,0.053578
49,4,0.05,0.8,100,-0.186330,1.068271,-2.031021,0.052524
23,5,0.05,0.8,150,-0.266665,1.089560,-1.833158,0.040150
26,6,0.05,1.0,150,-0.439102,0.993248,-1.805557,0.026748
52,7,0.05,1.0,100,-0.445314,0.867060,-2.027600,0.051665
20,8,0.05,0.5,150,-0.460489,0.756990,-1.732528,0.041675
46,9,0.05,0.5,100,-0.547172,0.823692,-1.862301,0.040175
0,10,0.01,0.5,50,-0.587496,0.114173,-0.648593,0.028831


In [832]:
# Strong NGB parameters

strong_ngb['best_params']

{'Base': DecisionTreeRegressor(criterion='friedman_mse', max_depth=3, min_samples_leaf=8,
                       min_samples_split=15),
 'Dist': ngboost.distns.normal.Normal,
 'learning_rate': 0.05,
 'minibatch_frac': 0.8,
 'n_estimators': 150}

### Feature Importance for a Strong Performing NGBoost Model

In [833]:
# Get feature importance

strong_ngb_model = strong_ngb['best_estimator']

feature_importance_loc = strong_ngb_model.feature_importances_[0]    # Mean
feature_importance_scale = strong_ngb_model.feature_importances_[1]  # Standard deviation

In [834]:
# Create feature importance data frames

X_test_df = pd.DataFrame(strong_ngb['X_test']).reset_index(drop=True)

features = X_test_df.columns

loc_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance_loc * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

scale_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance_scale * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

In [835]:
# Plot feature importance for location

fig_loc = px.bar(
    loc_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong NGB Top 10 Feature Importances: Location (μ)</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_RED]
)

fig_loc.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig_loc.show()

In [836]:
# Plot feature importance for scale

fig_scale = px.bar(
    scale_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong NGB Top 10 Feature Importances: Scale (σ)</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_NAVY]
)

fig_scale.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig_scale.show()

## XGBoost Model for Price Change Prediction

In [837]:
# Load in all XGBoost results across different double binned targets

xgb_dir_path = "../models/xgb/"

xgb_results = {}
for w in range(1, 7):
    for b in range(1, 7):
        file_name = f"model_d_px_chg_w{w}_b{b}"
        path = Path(xgb_dir_path) / f"{file_name}.pkl"
        try:
            with open(path, 'rb') as f:
                xgb_results[file_name] = pickle.load(f)
        except FileNotFoundError:
            print(f"Missing: {path}")

In [838]:
# Get summary statistics for plotting

xgb_model_stats = []
for k, v in xgb_results.items():
    xgb_model_stats.append({
        'y_col'       : k,
        'window'      : k.split('_')[-2][1],
        'bin'         : k.split('_')[-1][1],
        'test_r2'     : xgb_results[k]['best_test_r2'],
        'overfitting' : xgb_results[k]['best_overfitting']
    })

xgb_model_stats_df = pd.DataFrame(xgb_model_stats)

xgb_model_stats_df.sort_values('test_r2', ascending=False).head()

,y_col,window,bin,test_r2,overfitting
23,model_d_px_chg_w4_b6,4,6,0.594121,-0.085816
33,model_d_px_chg_w6_b4,6,4,0.581331,-0.067455
9,model_d_px_chg_w2_b4,2,4,0.573105,-0.063028
19,model_d_px_chg_w4_b2,4,2,0.572411,-0.072448
21,model_d_px_chg_w4_b4,4,4,0.571898,-0.060253


In [839]:
# Plotting XGBoost models across all window and bin combinations

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=xgb_model_stats_df['window'],
    y=xgb_model_stats_df['bin'],
    z=xgb_model_stats_df['test_r2'],
    mode='markers',
    marker=dict(
        size=8,
        color=xgb_model_stats_df['overfitting'],
        colorscale=laa_color_scale,
        opacity=0.9,
        colorbar=dict(
            title=dict(
                text='Test R² - Train R²',
                font=dict(color=LAA_SILVER)
            ),
            tickfont=dict(color=LAA_SILVER)
        ),
        showscale=True
    ),
    text=[
        f"Window: {w}<br>Bin: {b}<br>Test R²: {r:.4f}<br>Overfitting: {m:.4f}" 
        for w, b, r, m in zip(
            xgb_model_stats_df['window'], 
            xgb_model_stats_df['bin'], 
            xgb_model_stats_df['test_r2'], 
            xgb_model_stats_df['overfitting']
            )
    ],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    title=dict(
        text='<b>Optimally Parameterized XGBoost Models (LAA)</b>',
        font={'size': 25, 'color': LAA_SILVER},
        x=0.5,
        xanchor='center',
        y=0.95,        
        yanchor='top' 
    ),
    scene=dict(
        xaxis_title='Window Size (α)',
        yaxis_title='Bin Size (β)',
        zaxis_title='Test R²',
        xaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        yaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        zaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        bgcolor='#111111',
        aspectmode='cube'
    ),
    width=1000,
    height=700,
    margin=dict(l=0, r=0, b=0, t=60),
    showlegend=False
)

fig.show()

### Parameter Analysis for a Strong Performing XGBoost Model

In [840]:
# Get strong performing XGB model, uses target with window = 4 and bin = 4 

strong_xgb = xgb_results["model_d_px_chg_w4_b4"]

In [841]:
# Strong XGB statistics

print('Train MSE:', strong_xgb['best_train_mse'])
print('Test MSE:', strong_xgb['best_test_mse'])
print('Train R²:', strong_xgb['best_train_r2'])
print('Test R²:', strong_xgb['best_test_r2'])

Train MSE: 0.014280222426376226
Test MSE: 0.009978312759054966
Train R²: 0.5116450647520461
Test R²: 0.5718984656988768


In [842]:
# Random Search CV results for target with window = 4 and bin = 4

rscv_cols = [
    'rank_test_score',
    'param_reg_lambda',   
    'param_reg_alpha', 
    'param_n_estimators', ''
    'param_min_child_weight', 
    'param_max_depth',    
    'param_learning_rate',
    'param_gamma',        
    'param_colsample_bytree', 
    'mean_train_score',   
    'std_train_score',
    'mean_test_score', 
    'std_test_score',
]

strong_xgb['cv_results'][rscv_cols].sort_values('rank_test_score')

,rank_test_score,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,param_learning_rate,param_gamma,param_colsample_bytree,mean_train_score,std_train_score,mean_test_score,std_test_score
6,1,10,1,50,15,2,0.30,0.5,0.5,-0.086914,0.002455,-0.088699,0.012248
0,2,10,1,200,5,2,0.10,0.5,0.5,-0.090598,0.001796,-0.092387,0.012500
42,3,50,1,100,10,2,0.10,0.5,0.5,-0.096642,0.002215,-0.098739,0.011051
49,4,50,1,200,10,2,0.05,0.5,0.7,-0.097015,0.001616,-0.099086,0.011796
38,5,100,1,100,15,2,0.10,0.5,0.5,-0.104669,0.001822,-0.106739,0.011448
1,6,100,1,200,10,2,0.05,0.5,0.7,-0.105177,0.001159,-0.107193,0.011949
15,6,100,1,50,15,3,0.05,0.5,0.7,-0.105177,0.001159,-0.107193,0.011949
19,8,10,1,100,10,3,0.01,1.0,0.7,-0.105404,0.001419,-0.107277,0.011857
45,9,10,1,50,5,2,0.01,1.0,0.5,-0.114189,0.002328,-0.115814,0.010782
14,10,50,1,200,10,3,0.10,1.0,0.5,-0.115893,0.001070,-0.117358,0.012602


In [843]:
# Strong XGB parameters

strong_xgb['best_params']

{'reg_lambda': 10,
 'reg_alpha': 1,
 'n_estimators': 50,
 'min_child_weight': 15,
 'max_depth': 2,
 'learning_rate': 0.3,
 'gamma': 0.5,
 'colsample_bytree': 0.5}

### Feature Importance for a Strong Performing XGBoost Model

In [844]:
# Get feature importance

strong_xgb_model = strong_xgb['best_estimator']

feature_importance = strong_xgb_model.feature_importances_

In [845]:
# Create feature importance data frames

X_test_df = pd.DataFrame(strong_xgb['X_test']).reset_index(drop=True)

features = X_test_df.columns

feature_importance_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

In [846]:
# Plot feature importance

fig = px.bar(
    feature_importance_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong XGB Top 10 Feature Importances</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_RED]
)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig.show()

## Model Performance on Real-World Examples

### Example One: The Angels Hit a Homer

[On August 6, 2025, the Angels (LAA) played the Rays (TB).](https://www.espn.com/mlb/game/_/gameId/401696618/rays-angels)

It's the 3rd inning and the score is 4-1 Rays. Mike Trout is up to bat with two runners on base. Checking your phone, you see that Kalshi's market for the game is trading at ~21¢ (LAA has a ~21% chance of winning). You look up, and Trout has just hit a home run! You wonder what the market will be trading at now?

In [847]:
# Get the specific LAA vs TB game from the test set

laa_vs_tb = X_test_df.loc[85, :] # Index of the specific game

In [848]:
# Price change according to the double binned method

px_at_hr = laa_vs_tb['d_yes_price_w4_b4']

dbm_px_chg = float(strong_ngb['y_test'][85]) # Index of the specific game

print(f"Double Binned Mean Price Before Home Run: {px_at_hr*100:.2f}¢")
print(f"Double Binned Mean Price Change: {dbm_px_chg*100:.2f}¢")
print(f"Double Binned Mean Price After Home Run: {100 * (px_at_hr+dbm_px_chg):.2f}¢")

Double Binned Mean Price Before Home Run: 21.62¢
Double Binned Mean Price Change: 27.12¢
Double Binned Mean Price After Home Run: 48.75¢


### Analysis Using NGBoost

In [849]:
# Predict the price change distribution for the home run

laa_vs_tb_dist = strong_ngb_model.pred_dist(laa_vs_tb.values.reshape(1, -1))

ngb_mean_px_chg = laa_vs_tb_dist.params['loc'][0]
ngb_std_px_chg = laa_vs_tb_dist.params['scale'][0]

print(f"Strong NGB Expected Price Change: {ngb_mean_px_chg*100:.2f}¢")
print(f"Strong NGB Standard Deviation Price Change: {ngb_std_px_chg*100:.2f}¢")
print(f"Strong NGB Expected Price on Kalshi: {(px_at_hr+ngb_mean_px_chg)*100 :.2f}¢")

Strong NGB Expected Price Change: 18.38¢
Strong NGB Standard Deviation Price Change: 2.85¢
Strong NGB Expected Price on Kalshi: 40.00¢


In [850]:
# Confidence intervals for price change

z99 = norm.ppf(0.995)
z95 = norm.ppf(0.975)

ci99 = np.array((
    ngb_mean_px_chg - ngb_std_px_chg * z99,
    ngb_mean_px_chg + ngb_std_px_chg * z99
))
ci95 = np.array((
    ngb_mean_px_chg - ngb_std_px_chg * z95,
    ngb_mean_px_chg + ngb_std_px_chg * z95
))

price99 = (px_at_hr + ci99)
price95 = (px_at_hr + ci95)

print("Strong NGB Expected Price Change:")
print(f"99% CI: [{ci99[0]*100:.2f}¢, {ci99[1]*100:.2f}¢]")
print(f"95% CI: [{ci95[0]*100:.2f}¢, {ci95[1]*100:.2f}¢]")
print()
print("Strong NGB Expected Price on Kalshi:")
print(f"99% CI: [{price99[0]*100:.2f}¢, {price99[1]*100:.2f}¢]")
print(f"95% CI: [{price95[0]*100:.2f}¢, {price95[1]*100:.2f}¢]")

Strong NGB Expected Price Change:
99% CI: [11.03¢, 25.73¢]
95% CI: [12.79¢, 23.97¢]

Strong NGB Expected Price on Kalshi:
99% CI: [32.66¢, 47.35¢]
95% CI: [34.41¢, 45.59¢]


### Analysis Using XGBoost

In [851]:
# Predict the price change for the home run

xgb_px_chg = strong_xgb_model.predict(laa_vs_tb.values.reshape(1, -1))

print(f"Strong XGB Expected Price Change: {xgb_px_chg[0]*100:.2f}¢")
print(f"Strong XGB Expected Price on Kalshi: {(px_at_hr+xgb_px_chg[0])*100:.2f}¢")

Strong XGB Expected Price Change: 7.15¢
Strong XGB Expected Price on Kalshi: 28.77¢


### Real-World Outcome

In [852]:
# Instantiate DataProcessing object for LAA

laa_DP = DataProcessing(
    team = 'LAA',
    trade_path = "../data/raw/laa_kalshi_trade_data.parquet",
    score_path = "../data/raw/laa_score_data.parquet",
    homerun_path = "../data/raw/laa_homerun_data.parquet"
)

laa_trade_df = laa_DP.get_trade_df()

In [853]:
# Load LAA vs TB August 6, 2025 game trades

laa_vs_tb_ticker = "KXMLBGAME-25AUG06TBLAA-LAA"
laa_vs_tb_trades_df = laa_trade_df[laa_trade_df['ticker'] == laa_vs_tb_ticker].reset_index(drop=True).copy()

laa_vs_tb_trades_df.head()

,ticker,time_pst,yes_price_dollars,trade_count
0,KXMLBGAME-25AUG06TBLAA-LAA,2025-08-06 06:07:13.284820-07:00,0.48,201.0
1,KXMLBGAME-25AUG06TBLAA-LAA,2025-08-06 06:10:51.847312-07:00,0.48,221.0
2,KXMLBGAME-25AUG06TBLAA-LAA,2025-08-06 06:24:23.292600-07:00,0.48,25.0
3,KXMLBGAME-25AUG06TBLAA-LAA,2025-08-06 06:25:06.511524-07:00,0.48,24.0
4,KXMLBGAME-25AUG06TBLAA-LAA,2025-08-06 06:26:38.830150-07:00,0.48,10.0


In [854]:
# Plot LAA vs TB August 6, 2025 game

# Approximate home run time according to double binned mean method
laa_vs_tb_trades_df['time_pst'] = pd.to_datetime(laa_vs_tb_trades_df['time_pst'])
homerun_time = pd.to_datetime('2025-08-06 14:03:17.161322-07:00')

# Prepare data for plotting
start_time = homerun_time - timedelta(minutes=15)
end_time = homerun_time + timedelta(minutes=15)

game_window = laa_vs_tb_trades_df[
    (laa_vs_tb_trades_df['time_pst'] >= start_time) & 
    (laa_vs_tb_trades_df['time_pst'] <= end_time)
].copy()

game_window['yes_price_dollars'] = game_window['yes_price_dollars'] * 100

hr_price = game_window.loc[game_window['time_pst'] == homerun_time, 'yes_price_dollars'].iloc[0]

# Plotting
fig = px.line(game_window, x='time_pst', y='yes_price_dollars', color_discrete_sequence=[LAA_RED])

fig.update_layout(
    template='plotly_dark',
    title={
        'text': "<b>Kalshi Market Reaction to Mike Trout's Home Run</b><br><span style='font-size:16px; color:" + LAA_SILVER + "'>LAA vs TB | August 6, 2025</span>",
        'font': {'size': 20, 'color': LAA_SILVER},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis_title='Time (PST)',
    yaxis_title='Price (Probability of LAA Winning)',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.add_annotation(
    x=homerun_time,
    y=hr_price,
    text="<b>HR is Hit</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor='white',
    font=dict(color='white', size=12),
    bgcolor='rgba(186, 0, 33, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2
)

fig.add_annotation(
    text=f"<b>Expected Price on Kalshi After Home Run</b><br>" + 
         f"Strong NGB: {(px_at_hr + ngb_mean_px_chg) * 100 :.0f} <br>" +
         f"Strong XGB: {(px_at_hr + xgb_px_chg[0]) * 100 :.0f}",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12, color='white'),
    bgcolor='rgba(17, 17, 17, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2,
    borderpad=10
)

fig.show()

### Example Two: An Angels Opponent Hits a Homer

[On May 14, 2025, the Angels (LAA) played the Padres's (SD).](https://www.espn.com/mlb/game/_/gameId/401695554/angels-padres) 

It's the bottom of the 1st, and the score is tied 0-0. Xander Bogaerts is up to bat with two runners on base. Checking your phone, you see that Kalshi's market for the game is trading at ~35¢ (LAA has a ~35% chance of winning). You look up, and Bogaerts has just hit a home run! You wonder what the market will be trading at now?

In [855]:
# Get the specific LAA vs SD game from the test set

laa_vs_sd = X_test_df.loc[2, :] # Index of the specific game

In [856]:
# Price change according to the double binned method

px_at_hr = laa_vs_sd['d_yes_price_w4_b4']

dbm_px_chg = float(strong_ngb['y_test'][2]) # Index of the specific game

print(f"Double Binned Mean Price Before Home Run: {px_at_hr*100:.2f}¢")
print(f"Double Binned Mean Price Change: {dbm_px_chg*100:.2f}¢")
print(f"Double Binned Mean Price After Home Run: {100*(px_at_hr+dbm_px_chg):.2f}¢")

Double Binned Mean Price Before Home Run: 33.50¢
Double Binned Mean Price Change: -18.00¢
Double Binned Mean Price After Home Run: 15.50¢


### Analysis Using NGBoost

In [857]:
# Predict the price change distribution for the home run

laa_vs_sd_dist = strong_ngb_model.pred_dist(laa_vs_sd.values.reshape(1, -1))

ngb_mean_px_chg = laa_vs_sd_dist.params['loc'][0]
ngb_std_px_chg = laa_vs_sd_dist.params['scale'][0]

print(f"Strong NGB Expected Price Change: {ngb_mean_px_chg*100:.2f}¢")
print(f"Strong NGB Standard Deviation Price Change: {ngb_std_px_chg*100:.2f}¢")
print(f"Strong NGB Expected Price on Kalshi: {(px_at_hr+ngb_mean_px_chg)*100 :.2f}¢")

Strong NGB Expected Price Change: -15.63¢
Strong NGB Standard Deviation Price Change: 3.63¢
Strong NGB Expected Price on Kalshi: 17.87¢


In [858]:
# Confidence intervals for price change

z99 = norm.ppf(0.995)
z95 = norm.ppf(0.975)

ci99 = np.array((
    ngb_mean_px_chg - ngb_std_px_chg * z99,
    ngb_mean_px_chg + ngb_std_px_chg * z99
))
ci95 = np.array((
    ngb_mean_px_chg - ngb_std_px_chg * z95,
    ngb_mean_px_chg + ngb_std_px_chg * z95
))

price99 = (px_at_hr + ci99)
price95 = (px_at_hr + ci95)

print("Strong NGB Expected Price Change:")
print(f"99% CI: [{ci99[0]*100:.2f}¢, {ci99[1]*100:.2f}¢]")
print(f"95% CI: [{ci95[0]*100:.2f}¢, {ci95[1]*100:.2f}¢]")
print()
print("Strong NGB Expected Price on Kalshi:")
print(f"99% CI: [{price99[0]*100:.2f}¢, {price99[1]*100:.2f}¢]")
print(f"95% CI: [{price95[0]*100:.2f}¢, {price95[1]*100:.2f}¢]")

Strong NGB Expected Price Change:
99% CI: [-24.98¢, -6.29¢]
95% CI: [-22.74¢, -8.52¢]

Strong NGB Expected Price on Kalshi:
99% CI: [8.52¢, 27.21¢]
95% CI: [10.76¢, 24.98¢]


### Analysis Using XGBoost

In [859]:
# Predict the price change for the home run

xgb_px_chg = strong_xgb_model.predict(laa_vs_sd.values.reshape(1, -1))

print(f"Strong XGB Expected Price Change: {xgb_px_chg[0]*100:.2f}¢")
print(f"Strong XGB Expected Price on Kalshi: {(px_at_hr+xgb_px_chg[0])*100:.2f}¢")

Strong XGB Expected Price Change: -8.32¢
Strong XGB Expected Price on Kalshi: 25.18¢


### Real-World Outcome

In [860]:
# Load LAA vs SD May 14, 2025 game trades

laa_vs_sd_ticker = "KXMLBGAME-25MAY14LAASD-LAA"
laa_vs_sd_trades_df = laa_trade_df[laa_trade_df['ticker'] == laa_vs_sd_ticker].reset_index(drop=True).copy()

laa_vs_sd_trades_df.head()

,ticker,time_pst,yes_price_dollars,trade_count
0,KXMLBGAME-25MAY14LAASD-LAA,2025-05-14 05:14:39.892094-07:00,0.38,5.0
1,KXMLBGAME-25MAY14LAASD-LAA,2025-05-14 05:34:32.917609-07:00,0.38,50.0
2,KXMLBGAME-25MAY14LAASD-LAA,2025-05-14 05:48:58.793478-07:00,0.38,100.0
3,KXMLBGAME-25MAY14LAASD-LAA,2025-05-14 06:00:25.132002-07:00,0.38,12.0
4,KXMLBGAME-25MAY14LAASD-LAA,2025-05-14 06:31:36.470026-07:00,0.38,3.0


In [861]:
# Plot LAA vs SD May 14, 2025 game

# Approximate home run time according to double binned mean method
laa_vs_sd_trades_df['time_pst'] = pd.to_datetime(laa_vs_sd_trades_df['time_pst'])
homerun_time = pd.to_datetime('2025-05-14 18:53:47.426224-07:00')

# Prepare data for plotting
start_time = homerun_time - timedelta(minutes=15)
end_time = homerun_time + timedelta(minutes=15)

game_window = laa_vs_sd_trades_df[
    (laa_vs_sd_trades_df['time_pst'] >= start_time) & 
    (laa_vs_sd_trades_df['time_pst'] <= end_time)
].copy()

game_window['yes_price_dollars'] = game_window['yes_price_dollars'] * 100

hr_price = game_window.loc[game_window['time_pst'] == homerun_time, 'yes_price_dollars'].iloc[0]

# Plotting
fig = px.line(game_window, x='time_pst', y='yes_price_dollars', color_discrete_sequence=[LAA_RED])

fig.update_layout(
    template='plotly_dark',
    title={
        'text': "<b>Kalshi Market Reaction to Xander Bogaert's Home Run</b><br><span style='font-size:16px; color:" + LAA_SILVER + "'>LAA vs SD | May 14, 2025</span>",
        'font': {'size': 20, 'color': LAA_SILVER},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis_title='Time (PST)',
    yaxis_title='Price (Probability of LAA Winning)',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.add_annotation(
    x=homerun_time,
    y=hr_price,
    text="<b>HR is Hit</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor='white',
    font=dict(color='white', size=12),
    bgcolor='rgba(186, 0, 33, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2
)

fig.add_annotation(
    text=f"<b>Expected Price on Kalshi After Home Run</b><br>" + 
         f"Strong NGB: {(px_at_hr + ngb_mean_px_chg) * 100 :.2f}¢ <br>" +
         f"Strong XGB: {(px_at_hr + xgb_px_chg[0]) * 100 :.2f}¢",
    xref="paper", yref="paper",
    x=0.02, y=0.40,
    xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12, color='white'),
    bgcolor='rgba(17, 17, 17, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2,
    borderpad=10
)

fig.show()

## Summary of Predictive Modeling

### Target

- Recall, the Double Binned Mean method was used to determine the target being predicted in this analysis. The reader may wonder, why? This method solves two problems.

    - Problem One: Data Quality

        - The window parameter, denoted $\alpha$, determines the number of trades to go out prior or after a home run was claimed to have been hit by `mlbstatsapi`. From the [data analysis](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/data_analysis.ipynb) section it was determined that `mlbstatsapi` provides lagged data, with home runs and scores matched to timestamps after they occurred. The window parameter corrects this issue.

    - Problem Two: Kalshi Market Liquidity

        - The bin parameter, denoted $\beta$, determines the number of trades to include in the first window (price window before the home run was hit) and the second window (price window after the home run was hit). Simply taking one trade before and after a home run may result in a poor representation of the market price. Instead, an average could capture a fair market price.

- This approach is further justified by the results in the [data analysis](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/data_analysis.ipynb) and [hypothesis testing](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/hypothesis_testing.ipynb) notebooks.

- The models trained using **d_px_chg_w4_b4** as the target was chosen for further analysis due to its strong performance in both NGBoost and XGBoost models.

### NGBoost Models

- NGBoost models displayed strong performance with the top five best performing models by test $R^2$ ranging from 0.82 to 0.85, and overfitting ranging from 0.11 to 0.14 for those same models (measured by train $R^2$ minus test $R^2$).

- The parameters selected for the model using **d_px_chg_w4_b4** via Grid Search Cross Validation were:

    - Distribution: Normal

    - Base: Decision Tree Regressor

        - Criterion: Friedman's MSE

        - Max Depth: 3

        - Minimum Samples per Leaf: 8

        - Minimum Samples per Split: 15

    - Learning Rate: 0.05

    - Minibatch Fraction: 0.8

    - Number of Trees (estimators): 150

### XGBoost Models

- XGBoost models displayed strong performance with the top five best performing models by test $R^2$ ranging from 0.57 to 0.59, and overfitting ranging from -0.06 to -0.09 for those same models (measured by train $R^2$ minus test $R^2$). Noticeably less overfitting than NGBoost models, which can be attributed to the regularization imposed on XGBoost models.

- The parameters selected for the model using **d_px_chg_w4_b4** via Random Search Cross Validation were:

    - L1 Regularization (alpha): 1

    - L2 Regularization (lambda): 10

    - Minimum Loss Reduction for Further Partitioning (gamma): 0.5

    - Column Subsample by Tree: 0.5

    - Minimum Child Weight: 15

    - Maximum Tree Depth: 2

    - Learning Rate: 0.3

    - Number of Trees (estimators): 50